# Tree-based model for creating weights from spatial data

We use the exact coordinates of the listings to train a tree-based model that creates a price map of the city. For each AirBnB the model outputs an embedded categorial variable that can be used as an input for neural learning. We compare this approach to the direct use of neighbourhood data in order to quantify the effectiveness of the tree-based model in capturing spatial information.

In [44]:
from math import log1p

# Imports:

import import_ipynb
import numpy as np
import pandas as pd
import geopandas as gpd
import sklearn
from general import load_listings
from scipy.sparse import csr_matrix

listings_df = load_listings()
neighbourhoods_df = pd.read_csv("data/neighbourhoods.csv")
neighbourhoods_gdf = gpd.read_file("data/neighbourhoods.geojson")

## Preprocessing spatial data

For later sanity checks we will use the actual neighbourhood's influence. Also, we prepare the data for later visualization.

In [45]:
# Removing outliers:
listings_df.drop(listings_df[listings_df["price"] > 1000].index, inplace=True)

# One-hot encoding neighbourhoods:
n = len(neighbourhoods_df)
nbh_encoding = dict()
for i, nbh in enumerate(neighbourhoods_df["neighbourhood"]):
    nbh_encoding[nbh] = csr_matrix(([1], ([0], [i])), shape=(1, n))
listings_df["nbh_one_hot"] = listings_df["neighbourhood_cleansed"].map(nbh_encoding)

# Adding geometry:
listings_df["position"] = gpd.points_from_xy(listings_df.longitude, listings_df.latitude)

In [46]:
# Prepare data for growing tree:
X = listings_df[["longitude", "latitude"]].to_numpy()
y = listings_df["price"].to_numpy()
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(X, y, test_size=0.2, random_state=42)

## Growing the tree

In [47]:
def grow_tree(X_train, y_train):
    model = sklearn.tree.DecisionTreeRegressor(max_leaf_nodes=50, random_state=42)
    return model.fit(X_train, y_train)

In [48]:
def validate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    std = y_test.std()
    rmse = np.sqrt(sklearn.metrics.mean_squared_error(y_test, y_pred))
    mae = sklearn.metrics.mean_absolute_error(y_test, y_pred)
    r2 = sklearn.metrics.r2_score(y_test, y_pred)
    print(f"RMSE: {rmse} (std: {std})")
    print(f"MAE: {mae}")
    print(f"R2: {r2}")

In [49]:
proto = grow_tree(X_train, y_train)
validate(proto, X_test, y_test)

RMSE: 82.9513859603807 (std: 85.69513166875714)
MAE: 46.34795329594413
R2: 0.06300992103972647


In [50]:
pd.DataFrame(proto.apply(X_test))

,0
0,55
1,43
2,12
3,43
4,14
...,...
1946,95
1947,55
1948,95
1949,55


## Cost complexity pruning

In [52]:
ccp_model = grow_tree(X_train, y_train)
path = ccp_model.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities   # Gini

# Train trees with different pruning levels
trees = []
for ccp_alpha in ccp_alphas:
    tree_pruned = sklearn.tree.DecisionTreeRegressor(random_state=42, ccp_alpha=ccp_alpha)
    tree_pruned.fit(X_train, y_train)
    trees.append(tree_pruned)

# Evaluate each on test set
rmses = []
for tree_pruned in trees:
    y_pred = tree_pruned.predict(X_test)
    rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))
    rmses.append(rmse)

# Find best alpha
best_idx = np.argmin(rmses)
best_ccp_alpha = ccp_alphas[best_idx]
best_rmse = rmses[best_idx]

print(f"Best ccp_alpha: {best_ccp_alpha:.4f}")
print(f"Best test RMSE: {best_rmse:.2f}")


Best ccp_alpha: 50.9544
Best test RMSE: 84.62
